# DreamGuard — Study Analysis

**Setup:**
```bash
cd scripts
uv sync                   # install dependencies
```

**Pull data from Quest:**
```bash
adb pull /sdcard/Android/data/com.DefaultCompany.DreamGuard/files/experiments ../data
```

Expected layout:
```
data/
  study_1/  study.csv  rooms.csv  collection.csv  position.csv  headset.csv  r_controller.csv  l_controller.csv
  study_2/  ...
```

In [1]:
from pathlib import Path
import warnings
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from matplotlib.lines import Line2D
from PIL import Image
from IPython.display import display, Image as IPyImage

warnings.filterwarnings('ignore')
%matplotlib inline

SCRIPTS_DIR = Path('.').resolve()
DATA_DIR    = SCRIPTS_DIR.parent / 'data'

# Check playground output location first, then docs/assets fallback
_img_candidates = [
    SCRIPTS_DIR.parent / 'data' / 'dungeon' / 'DungeonTopDown.png',
    SCRIPTS_DIR.parent / 'docs' / 'assets' / 'DungeonTopDown.png',
]
DUNGEON_IMG  = next((p for p in _img_candidates if p.exists()), _img_candidates[-1])
DUNGEON_JSON = DUNGEON_IMG.with_suffix('.json')

assert DATA_DIR.exists(), f'data/ not found at {DATA_DIR}'
sessions = sorted(DATA_DIR.glob('study_*'), key=lambda p: int(p.name.split('_')[1]))
assert sessions, 'No study_* folders found'
print(f'Found {len(sessions)} session(s): {[s.name for s in sessions]}')
print(f'Dungeon image : {DUNGEON_IMG}  (exists={DUNGEON_IMG.exists()})')
print(f'Dungeon JSON  : {DUNGEON_JSON} (exists={DUNGEON_JSON.exists()})')

Found 1 session(s): ['study_20']
Dungeon image : F:\Unity\Projects\DreamGuard\docs\assets\DungeonTopDown.png  (exists=True)
Dungeon JSON  : F:\Unity\Projects\DreamGuard\docs\assets\DungeonTopDown.json (exists=False)


## Load all sessions

In [2]:
def read_csv(session_dir: Path, name: str) -> pd.DataFrame:
    p = session_dir / name
    if not p.exists():
        return pd.DataFrame()
    df = pd.read_csv(p)
    if 'timestamp_iso' in df.columns:
        df['timestamp_iso'] = pd.to_datetime(df['timestamp_iso'])
    return df

def extract(detail: str, key: str) -> str:
    if pd.isna(detail):
        return ''
    for part in str(detail).split():
        if part.startswith(f'{key}='):
            return part[len(key)+1:]
    return ''

def nearest_pos(pos_df: pd.DataFrame, ts: pd.Timestamp):
    """Return the (x, z) position sample closest to timestamp ts."""
    if pos_df.empty:
        return None, None
    idx = (pos_df['timestamp_iso'] - ts).abs().idxmin()
    r = pos_df.loc[idx]
    return r['x'], r['z']

data = {}
for sd in sessions:
    data[sd.name] = {
        'dir':        sd,
        'study':      read_csv(sd, 'study.csv'),
        'rooms':      read_csv(sd, 'rooms.csv'),
        'collection': read_csv(sd, 'collection.csv'),
        'position':   read_csv(sd, 'position.csv'),
        'headset':    read_csv(sd, 'headset.csv'),
        'r_ctrl':     read_csv(sd, 'r_controller.csv'),
        'l_ctrl':     read_csv(sd, 'l_controller.csv'),
    }

print('Loaded.')

Loaded.


## Session Overview

In [3]:
rows = []
for name, d in data.items():
    s = d['study']
    if s.empty:
        continue
    t0 = s[s.event_type == 'SESSION_START']['timestamp_iso']
    t1 = s[s.event_type == 'SESSION_END']['timestamp_iso']
    t0 = t0.iloc[0] if not t0.empty else s.timestamp_iso.min()
    t1 = t1.iloc[0] if not t1.empty else s.timestamp_iso.max()
    rows.append({
        'session':         name,
        'participant':     s['participant_id'].dropna().iloc[0],
        'condition':       s['condition'].dropna().iloc[0],
        'duration_min':    round((t1 - t0).total_seconds() / 60, 1),
        'rooms_complete':  (s.event_type == 'ROOM_COMPLETE').sum(),
        'orbs_collected':  len(d['collection']),
        'triggers':        (s.event_type == 'TRIGGER').sum(),
        'intrusions':      (s.event_type == 'INTRUSION_MARK').sum(),
        'false_positives': (s.event_type == 'FALSE_POSITIVE').sum(),
        'pos_samples':     len(d['position']),
    })

overview = pd.DataFrame(rows)
display(overview)

,session,participant,condition,duration_min,rooms_complete,orbs_collected,triggers,intrusions,false_positives,pos_samples
0,study_20,P00,baseline,0.5,0,0,1,0,0,211


## Trigger Latency

In [4]:
latency_rows = []
for name, d in data.items():
    s = d['study']
    if s.empty:
        continue
    halfways = s[s.event_type == 'ROOM_HALFWAY'].copy()
    triggers  = s[s.event_type == 'TRIGGER'].copy()
    halfways['room_id'] = halfways['detail'].apply(lambda x: extract(x, 'room_id'))
    pid  = s['participant_id'].dropna().iloc[0]
    cond = s['condition'].dropna().iloc[0]
    for _, hw in halfways.iterrows():
        later = triggers[triggers.timestamp_iso > hw.timestamp_iso]
        if later.empty:
            continue
        trig = later.iloc[0]
        latency_rows.append({
            'session': name, 'participant': pid, 'condition': cond,
            'room_id': hw['room_id'],
            'latency_s': (trig.timestamp_iso - hw.timestamp_iso).total_seconds(),
            'technique': extract(trig['detail'], 'technique'),
        })

latency_df = pd.DataFrame(latency_rows)

if latency_df.empty:
    print('No trigger latency data yet.')
else:
    display(latency_df)
    conditions = latency_df['condition'].unique()
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    groups = [latency_df[latency_df.condition == c]['latency_s'].values for c in conditions]
    axes[0].boxplot(groups, labels=conditions, patch_artist=True)
    axes[0].set_ylabel('Latency (s)')
    axes[0].set_title('Trigger Latency by Condition')
    axes[0].grid(axis='y', linestyle='--', alpha=0.5)

    for c in conditions:
        sub = latency_df[latency_df.condition == c]
        axes[1].scatter(sub['room_id'], sub['latency_s'], label=c, alpha=0.8, s=60)
    axes[1].set_ylabel('Latency (s)')
    axes[1].set_title('Trigger Latency per Room')
    axes[1].legend()
    axes[1].grid(axis='y', linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.savefig(DATA_DIR / 'latency.png', dpi=150, bbox_inches='tight')
    plt.show()

No trigger latency data yet.


## Room Durations

In [5]:
dur_rows = []
for name, d in data.items():
    rooms = d['rooms']
    s     = d['study']
    if rooms.empty or s.empty:
        continue
    cond = s['condition'].dropna().iloc[0]
    pid  = s['participant_id'].dropna().iloc[0]
    enters = rooms[rooms.event == 'ENTER'].set_index('room_id')['timestamp_iso']
    exits  = rooms[rooms.event == 'EXIT'].set_index('room_id')['timestamp_iso']
    for room_id in enters.index.intersection(exits.index):
        dur_rows.append({
            'session': name, 'participant': pid, 'condition': cond,
            'room_id': room_id,
            'duration_s': (exits[room_id] - enters[room_id]).total_seconds(),
        })

dur_df = pd.DataFrame(dur_rows)

if dur_df.empty:
    print('No room duration data yet.')
else:
    display(dur_df)
    conditions = dur_df['condition'].unique()
    room_ids   = sorted(dur_df['room_id'].unique())
    x = np.arange(len(room_ids))
    w = 0.8 / max(len(conditions), 1)

    fig, ax = plt.subplots(figsize=(10, 4))
    for i, cond in enumerate(conditions):
        sub   = dur_df[dur_df.condition == cond].groupby('room_id')['duration_s']
        means = [sub.mean().get(r, 0) for r in room_ids]
        stds  = [sub.std().get(r, 0)  for r in room_ids]
        ax.bar(x + i*w, means, w*0.9, yerr=stds, label=cond, capsize=4, alpha=0.8)
    ax.set_xticks(x + w*(len(conditions)-1)/2)
    ax.set_xticklabels(room_ids)
    ax.set_ylabel('Duration (s)')
    ax.set_title('Room Duration by Condition (mean ± SD)')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(DATA_DIR / 'room_durations.png', dpi=150, bbox_inches='tight')
    plt.show()

No room duration data yet.


## Minimap Setup

World-space bounds are auto-derived from position data (Unity X = horizontal, Z = depth / up in top-down view).

Each room panel uses **square display bounds** (`ROOM_DISPLAY`) centered on the room's trajectory centroid, with a uniform half-size across all rooms. This prevents squishing regardless of per-room Z extent variation. `ROOM_BOUNDS` is kept unchanged for data filtering.

- `_BG_ZOOM_FRAC` — extra fractional padding beyond the max trajectory extent (default 0.05 = 5%, shows outer walls)

If room_4's background is cut off on the right, the dungeon screenshot doesn't cover Z > 73.75 — re-run **Tools > DreamGuard > Dungeon Top-Down Screenshot** in Unity to update it.

In [6]:
dungeon_img = None
if DUNGEON_IMG.exists():
    dungeon_img = Image.open(DUNGEON_IMG).convert('RGBA')
    print(f'Dungeon image: {dungeon_img.size[0]}×{dungeon_img.size[1]} px  ({DUNGEON_IMG})')
else:
    print(f'WARNING: Dungeon image not found at {DUNGEON_IMG}')

# ── World bounds from DungeonTopDown.json ────────────────────────────────────
if DUNGEON_JSON.exists():
    cam = json.loads(DUNGEON_JSON.read_text())
    ortho   = cam['ortho_size']
    aspect  = cam['aspect']
    WORLD_XMIN = cam['center_x'] - ortho * aspect
    WORLD_XMAX = cam['center_x'] + ortho * aspect
    WORLD_ZMIN = cam['center_z'] - ortho
    WORLD_ZMAX = cam['center_z'] + ortho
    print(f'Camera params: center=({cam["center_x"]:.2f}, {cam["center_z"]:.2f})  orthoSize={ortho:.2f}')
else:
    print('WARNING: DungeonTopDown.json not found — re-run Tools > DreamGuard > Dungeon Top-Down Screenshot.')
    all_pos = pd.concat([d['position'] for d in data.values() if not d['position'].empty], ignore_index=True)
    if all_pos.empty:
        WORLD_XMIN, WORLD_XMAX = -10.0, 10.0
        WORLD_ZMIN, WORLD_ZMAX = -5.0, 40.0
    else:
        pad_x = max((all_pos['x'].max() - all_pos['x'].min()) * 0.05, 1.0)
        pad_z = max((all_pos['z'].max() - all_pos['z'].min()) * 0.05, 1.0)
        WORLD_XMIN = all_pos['x'].min() - pad_x; WORLD_XMAX = all_pos['x'].max() + pad_x
        WORLD_ZMIN = all_pos['z'].min() - pad_z; WORLD_ZMAX = all_pos['z'].max() + pad_z

EXTENT = [WORLD_XMIN, WORLD_XMAX, WORLD_ZMIN, WORLD_ZMAX]
print(f'World bounds — X: [{WORLD_XMIN:.2f}, {WORLD_XMAX:.2f}]  Z: [{WORLD_ZMIN:.2f}, {WORLD_ZMAX:.2f}]')

# ── Per-room bounds (used for data filtering) ─────────────────────────────────
def _compute_room_bounds_raw(data: dict) -> dict:
    """Derive X/Z extents for each room from timed position samples."""
    room_pos: dict[str, list] = {}
    for d in data.values():
        pos   = d['position']
        rooms = d['rooms']
        if pos.empty or rooms.empty:
            continue
        enters = rooms[rooms.event == 'ENTER'].set_index('room_id')['timestamp_iso']
        exits  = rooms[rooms.event == 'EXIT'].set_index('room_id')['timestamp_iso']
        for rid in enters.index.intersection(exits.index):
            mask = (pos.timestamp_iso >= enters[rid]) & (pos.timestamp_iso <= exits[rid])
            sub  = pos[mask]
            if not sub.empty:
                room_pos.setdefault(rid, []).append(sub)
    raw = {}
    for rid, dfs in room_pos.items():
        all_df = pd.concat(dfs)
        raw[rid] = dict(xmin=float(all_df.x.min()), xmax=float(all_df.x.max()),
                        zmin=float(all_df.z.min()), zmax=float(all_df.z.max()))
    return raw

_raw = _compute_room_bounds_raw(data)
SORTED_ROOMS = sorted(_raw, key=lambda r: _raw[r]['zmin'])

# Shared X extent across all rooms (with small padding)
_xpad = (max(v['xmax'] for v in _raw.values()) - min(v['xmin'] for v in _raw.values())) * 0.04
_ROOM_XMIN = min(v['xmin'] for v in _raw.values()) - _xpad
_ROOM_XMAX = max(v['xmax'] for v in _raw.values()) + _xpad

# Fill Z gaps between rooms: split at midpoints so panels tile seamlessly
_zmins = [_raw[r]['zmin'] for r in SORTED_ROOMS]
_zmaxs = [_raw[r]['zmax'] for r in SORTED_ROOMS]
_zpad  = (_zmaxs[-1] - _zmins[0]) * 0.02
_fzmins = [_zmins[0] - _zpad]
_fzmaxs = []
for i in range(len(SORTED_ROOMS) - 1):
    mid = (_zmaxs[i] + _zmins[i + 1]) / 2
    _fzmaxs.append(mid); _fzmins.append(mid)
_fzmaxs.append(_zmaxs[-1] + _zpad)

ROOM_BOUNDS = {
    rid: dict(xmin=_ROOM_XMIN, xmax=_ROOM_XMAX,
              zmin=_fzmins[i], zmax=_fzmaxs[i])
    for i, rid in enumerate(SORTED_ROOMS)
}
print(f'Room bounds ({len(SORTED_ROOMS)} rooms):')
for rid, rb in ROOM_BOUNDS.items():
    print(f'  {rid}: X=[{rb["xmin"]:.1f},{rb["xmax"]:.1f}]  Z=[{rb["zmin"]:.1f},{rb["zmax"]:.1f}]')

# ── Square display bounds (used for image crop and axes) ──────────────────────
# Each panel is a square centered on the room trajectory centroid.
# All panels use the same half-size = max(X_half, Z_half_max) × zoom factor.
# This ensures rooms appear undistorted regardless of per-room Z extent variation.
#
#   _BG_ZOOM_FRAC — extra fractional padding on each side (shows walls beyond trajectory)
_BG_ZOOM_FRAC = 0.05

_sq_x_half    = (_ROOM_XMAX - _ROOM_XMIN) / 2
_sq_z_half    = max((ROOM_BOUNDS[r]['zmax'] - ROOM_BOUNDS[r]['zmin']) / 2 for r in SORTED_ROOMS)
_sq_half      = max(_sq_x_half, _sq_z_half) * (1 + _BG_ZOOM_FRAC)
_sq_xc        = (_ROOM_XMIN + _ROOM_XMAX) / 2

ROOM_DISPLAY = {
    rid: dict(
        xmin = _sq_xc - _sq_half,
        xmax = _sq_xc + _sq_half,
        zmin = (ROOM_BOUNDS[rid]['zmin'] + ROOM_BOUNDS[rid]['zmax']) / 2 - _sq_half,
        zmax = (ROOM_BOUNDS[rid]['zmin'] + ROOM_BOUNDS[rid]['zmax']) / 2 + _sq_half,
    )
    for rid in SORTED_ROOMS
}
print(f'Square display half-size: {_sq_half:.2f} world units')
for rid, rd in ROOM_DISPLAY.items():
    print(f'  {rid}: X=[{rd["xmin"]:.1f},{rd["xmax"]:.1f}]  Z=[{rd["zmin"]:.1f},{rd["zmax"]:.1f}]')

# ── Per-room image helpers ────────────────────────────────────────────────────
_dungeon_arr = np.array(dungeon_img) if dungeon_img is not None else None

def _w2col(x):
    """World X → PIL image column index."""
    return int(np.clip((x - WORLD_XMIN) / (WORLD_XMAX - WORLD_XMIN) * 2048, 0, 2048))

def _w2row(z):
    """World Z → PIL image row (row 0 = top = high Z)."""
    return int(np.clip((1 - (z - WORLD_ZMIN) / (WORLD_ZMAX - WORLD_ZMIN)) * 2048, 0, 2048))

def _room_bg(rid: str):
    """
    Crop + rotate the dungeon image for one room so Z runs left→right.

    Uses ROOM_DISPLAY bounds (square, centered on room centroid) as the crop region,
    clamped to the image world bounds so pixel count always matches the displayed
    area — no squishing even if the room extends beyond the screenshot.

    Returns (rotated_array, imshow_extent) or (None, None).
    """
    if _dungeon_arr is None:
        return None, None
    rd = ROOM_DISPLAY[rid]
    eff_xmin = max(rd['xmin'], WORLD_XMIN); eff_xmax = min(rd['xmax'], WORLD_XMAX)
    eff_zmin = max(rd['zmin'], WORLD_ZMIN); eff_zmax = min(rd['zmax'], WORLD_ZMAX)
    if eff_xmax <= eff_xmin or eff_zmax <= eff_zmin:
        return None, None
    rt  = _w2row(eff_zmax); rb_ = _w2row(eff_zmin)
    cl  = _w2col(eff_xmin); cr  = _w2col(eff_xmax)
    crop = _dungeon_arr[rt:rb_, cl:cr]
    return np.rot90(crop, k=-1), [eff_zmin, eff_zmax, eff_xmin, eff_xmax]

def draw_bg_room(ax, rid: str):
    """Draw the dungeon background for one room with Z on the x-axis."""
    rd = ROOM_DISPLAY[rid]
    bg, extent = _room_bg(rid)
    if bg is not None:
        ax.imshow(bg, extent=extent, origin='lower', aspect='auto', zorder=0)
    ax.set_xlim(rd['zmin'], rd['zmax'])
    ax.set_ylim(rd['xmin'], rd['xmax'])

def make_room_fig(height: float = 3.5):
    """
    Figure with one square subplot per room, side by side with no spacing.
    All panels are the same size because ROOM_DISPLAY uses a uniform half-size.
    Returns (fig, axes).
    """
    n = len(SORTED_ROOMS)
    fig, axes = plt.subplots(1, n, figsize=(height * n, height),
                             gridspec_kw={'wspace': 0})
    if n == 1:
        axes = [axes]
    return fig, axes

def make_heatmap_room(dfs: list, rb: dict, resolution: int = 200) -> np.ndarray:
    """2-D histogram for one room with axes (x_bins, z_bins) — ready for imshow with Z horizontal."""
    xs = np.concatenate([df['x'].values for df in dfs])
    zs = np.concatenate([df['z'].values for df in dfs])
    mask = (xs >= rb['xmin']) & (xs <= rb['xmax']) & (zs >= rb['zmin']) & (zs <= rb['zmax'])
    xs, zs = xs[mask], zs[mask]
    if len(xs) == 0:
        return np.zeros((resolution, resolution))
    hm, _, _ = np.histogram2d(xs, zs, bins=resolution,
                               range=[[rb['xmin'], rb['xmax']], [rb['zmin'], rb['zmax']]])
    return hm  # shape (x_bins, z_bins); origin='lower' → bottom=low X, left=low Z ✓

def overlay_heatmap_room(ax, hm: np.ndarray, rb: dict, alpha: float = 0.6):
    masked = np.ma.masked_where(hm == 0, hm)
    cmap   = plt.cm.hot_r.copy(); cmap.set_bad(alpha=0.0)
    vmax   = max(hm.max(), 1)
    norm   = mcolors.PowerNorm(gamma=0.4, vmin=1, vmax=vmax)
    ax.imshow(masked, extent=[rb['zmin'], rb['zmax'], rb['xmin'], rb['xmax']],
              origin='lower', aspect='auto', cmap=cmap, alpha=alpha, zorder=1, norm=norm)
    return cmap, norm

EVENT_STYLES = {
    'ROOM_ENTER':       dict(marker='o',  color='#2196F3', ms=10, zorder=4, label='Room enter'),
    'ROOM_HALFWAY':     dict(marker='v',  color='#9C27B0', ms=10, zorder=4, label='Room halfway'),
    'ROOM_COMPLETE':    dict(marker='s',  color='#1565C0', ms=10, zorder=4, label='Room complete'),
    'INTRUSION_MARK':   dict(marker='^',  color='#F44336', ms=12, zorder=5, label='Intrusion'),
    'TRIGGER':          dict(marker='*',  color='#E91E63', ms=15, zorder=5, label='Trigger'),
    'FALSE_POSITIVE':   dict(marker='x',  color='#FF5722', ms=12, zorder=5, label='False positive'),
    'CONFEDERATE_EXIT': dict(marker='D',  color='#795548', ms=9,  zorder=4, label='Confederate exit'),
}

def legend_handle(etype: str) -> Line2D:
    st = EVENT_STYLES[etype]
    return Line2D([0],[0], marker=st['marker'], color='w',
                  markerfacecolor=st['color'], markersize=8,
                  label=st['label'], markeredgecolor='black', markeredgewidth=0.5)

print('Minimap setup complete.')

Dungeon image: 2048×2048 px  (F:\Unity\Projects\DreamGuard\docs\assets\DungeonTopDown.png)
World bounds — X: [-2.75, 7.41]  Z: [-25.06, 1.10]


ValueError: max() iterable argument is empty

## Minimap — Player Trajectory per Session

Path colour = time (purple → yellow). Symbols mark key events at the player's nearest recorded position.

In [ ]:
with_pos = [(n, d) for n, d in data.items() if not d['position'].empty]
if not with_pos:
    print('No position data.')
else:
    for name, d in with_pos:
        fig, axes = make_room_fig()
        study = d['study']
        pos   = d['position']
        xs, zs = pos['x'].values, pos['z'].values
        t = np.linspace(0, 1, len(xs))
        legend_pool = {}

        for ax, rid in zip(axes, SORTED_ROOMS):
            draw_bg_room(ax, rid)
            rb = ROOM_BOUNDS[rid]

            # Trajectory segment colour = time (plasma purple→yellow)
            mask = (zs >= rb['zmin']) & (zs <= rb['zmax'])
            rx, rz, rt = xs[mask], zs[mask], t[mask]
            for j in range(len(rx) - 1):
                ax.plot(rz[j:j+2], rx[j:j+2],
                        color=cm.plasma(rt[j]), lw=1.2, alpha=0.75, zorder=2)
            if len(rx):
                ax.plot(rz[0],  rx[0],  'o', color='#00E676', ms=9, zorder=6,
                        markeredgecolor='black', markeredgewidth=0.6, linestyle='None')
                ax.plot(rz[-1], rx[-1], 'X', color='#FF1744', ms=9, zorder=6,
                        markeredgecolor='black', markeredgewidth=0.6, linestyle='None')

            # Event markers — note swapped (z, x) order
            for etype, st in EVENT_STYLES.items():
                for _, ev in study[study.event_type == etype].iterrows():
                    px, pz = nearest_pos(pos, ev.timestamp_iso)
                    if px is None or not (rb['zmin'] <= pz <= rb['zmax']):
                        continue
                    ax.plot(pz, px, marker=st['marker'], color=st['color'],
                            markersize=st['ms'], zorder=st['zorder'],
                            markeredgecolor='black', markeredgewidth=0.5, linestyle='None')
                    legend_pool.setdefault(etype, legend_handle(etype))

            ax.set_title(rid, fontsize=8)
            ax.set_xlabel('Z', fontsize=7)
            if ax is axes[0]:
                ax.set_ylabel('X', fontsize=7)
            else:
                ax.set_yticklabels([])
                ax.tick_params(left=False)

        # Time colourbar on the right
        sm = plt.cm.ScalarMappable(cmap='plasma', norm=plt.Normalize(0, 1))
        sm.set_array([])
        fig.colorbar(sm, ax=axes[-1], label='Time', fraction=0.08, pad=0.02)

        extra = [
            Line2D([0],[0], marker='o', color='w', markerfacecolor='#00E676',
                   markersize=8, label='Start', markeredgecolor='black', markeredgewidth=0.5),
            Line2D([0],[0], marker='X', color='w', markerfacecolor='#FF1744',
                   markersize=8, label='End',   markeredgecolor='black', markeredgewidth=0.5),
        ]
        axes[0].legend(handles=extra + list(legend_pool.values()),
                       loc='upper left', fontsize=7, framealpha=0.85)

        pid  = study['participant_id'].dropna().iloc[0] if not study.empty else '?'
        cond = study['condition'].dropna().iloc[0]      if not study.empty else '?'
        fig.suptitle(f'{name}  P={pid}  cond={cond} — Player Trajectory per Room', fontsize=10, y=1.02)
        plt.savefig(DATA_DIR / f'trajectory_per_session.png', dpi=150, bbox_inches='tight')
        plt.show()

## Minimap — Heatmap Overlay (Combined)

In [ ]:
pos_dfs = [d['position'] for d in data.values() if not d['position'].empty]

if not pos_dfs:
    print('No position data.')
else:
    fig, axes = make_room_fig()
    cmap_last = norm_last = None

    for ax, rid in zip(axes, SORTED_ROOMS):
        draw_bg_room(ax, rid)
        hm = make_heatmap_room(pos_dfs, ROOM_BOUNDS[rid])
        cmap_last, norm_last = overlay_heatmap_room(ax, hm, ROOM_BOUNDS[rid])
        ax.set_title(rid, fontsize=8)
        ax.set_xlabel('Z', fontsize=7)
        if ax is axes[0]:
            ax.set_ylabel('X', fontsize=7)
        else:
            ax.set_yticklabels([])
            ax.tick_params(left=False)

    sm = plt.cm.ScalarMappable(cmap=cmap_last, norm=norm_last)
    sm.set_array([])
    fig.colorbar(sm, ax=axes[-1], label='Dwell time (samples)', fraction=0.08, pad=0.02)
    fig.suptitle(f'Combined Position Heatmap ({len(pos_dfs)} session(s))', fontsize=10, y=1.02)
    plt.savefig(DATA_DIR / 'heatmap_combined.png', dpi=150, bbox_inches='tight')
    plt.show()

## Minimap — Heatmap by Condition

Compare whether the active technique changes where participants spend time.

In [ ]:
cond_pos: dict[str, list] = {}
for name, d in data.items():
    s = d['study']
    if s.empty or d['position'].empty:
        continue
    cond = s['condition'].dropna().iloc[0]
    cond_pos.setdefault(cond, []).append(d['position'])

if not cond_pos:
    print('No data.')
else:
    conditions = sorted(cond_pos)
    from matplotlib.gridspec import GridSpec

    widths = [ROOM_BOUNDS[r]['zmax'] - ROOM_BOUNDS[r]['zmin'] for r in SORTED_ROOMS]
    x_ext  = _ROOM_XMAX - _ROOM_XMIN
    height = 3.5
    scale  = height / x_ext
    fig_w  = sum(w * scale for w in widths)
    fig    = plt.figure(figsize=(fig_w, height * len(conditions)))
    gs     = GridSpec(len(conditions), len(SORTED_ROOMS), figure=fig,
                      width_ratios=widths, wspace=0, hspace=0.45)

    for row, cond in enumerate(conditions):
        dfs = cond_pos[cond]
        for col, rid in enumerate(SORTED_ROOMS):
            ax = fig.add_subplot(gs[row, col])
            draw_bg_room(ax, rid)
            hm = make_heatmap_room(dfs, ROOM_BOUNDS[rid])
            cmap, norm = overlay_heatmap_room(ax, hm, ROOM_BOUNDS[rid])
            if col == 0:
                ax.set_ylabel(f'{cond}\nX', fontsize=7)
            else:
                ax.set_yticklabels([]); ax.tick_params(left=False)
            if row == 0:
                ax.set_title(rid, fontsize=8)
            ax.set_xlabel('Z', fontsize=7)

    # Single colourbar at top-right
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=fig.axes, label='Dwell time (samples)', fraction=0.015, pad=0.02)
    fig.suptitle('Position Heatmap by Condition', fontsize=12, y=1.01)
    plt.savefig(DATA_DIR / 'heatmap_by_condition.png', dpi=150, bbox_inches='tight')
    plt.show()

## Minimap — Intrusion → Trigger Positions

Where intrusions occurred and where the technique fired. Arrows connect the two events for the same episode.

In [ ]:
fig, axes = make_room_fig()
seen = set()

for ax, rid in zip(axes, SORTED_ROOMS):
    draw_bg_room(ax, rid)
    rb = ROOM_BOUNDS[rid]

    for name, d in data.items():
        s, pos = d['study'], d['position']
        if s.empty or pos.empty:
            continue

        for etype in ('INTRUSION_MARK', 'TRIGGER', 'FALSE_POSITIVE'):
            st = EVENT_STYLES[etype]
            for _, ev in s[s.event_type == etype].iterrows():
                px, pz = nearest_pos(pos, ev.timestamp_iso)
                if px is None or not (rb['zmin'] <= pz <= rb['zmax']):
                    continue
                lbl = st['label'] if (ax is axes[0] and etype not in seen) else '_nolegend_'
                seen.add(etype)
                ax.plot(pz, px, marker=st['marker'], color=st['color'],
                        markersize=st['ms'], zorder=st['zorder'],
                        markeredgecolor='black', markeredgewidth=0.6,
                        linestyle='None', label=lbl)

        # Arrow: intrusion → trigger
        intrusions = s[s.event_type == 'INTRUSION_MARK']
        triggers   = s[s.event_type == 'TRIGGER']
        for _, intr in intrusions.iterrows():
            later = triggers[triggers.timestamp_iso > intr.timestamp_iso]
            if later.empty:
                continue
            trig = later.iloc[0]
            ix, iz = nearest_pos(pos, intr.timestamp_iso)
            tx, tz = nearest_pos(pos, trig.timestamp_iso)
            if ix is not None and rb['zmin'] <= iz <= rb['zmax']:
                ax.annotate('', xy=(tz, tx), xytext=(iz, ix),
                            arrowprops=dict(arrowstyle='->', color='white', lw=1.5), zorder=7)

    ax.set_title(rid, fontsize=8)
    ax.set_xlabel('Z', fontsize=7)
    if ax is axes[0]:
        ax.set_ylabel('X', fontsize=7)
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(),
                  loc='upper left', fontsize=7)
    else:
        ax.set_yticklabels([]); ax.tick_params(left=False)

fig.suptitle('Intrusion → Trigger Positions', fontsize=10, y=1.02)
plt.savefig(DATA_DIR / 'minimap_intrusion_triggers.png', dpi=150, bbox_inches='tight')
plt.show()

## Minimap — Headset Gaze Direction

Arrows show where the participant was looking (XZ projection), sampled every N frames.

In [ ]:
def quat_forward_xz(qx, qy, qz, qw):
    """Rotate Unity's forward vector (0,0,1) by quaternion; return (fx, fz)."""
    fx = 2*(qx*qz + qw*qy)
    fz = 2*(qy*qz - qw*qx)
    mag = np.sqrt(fx**2 + fz**2) + 1e-9
    return fx/mag, fz/mag

ARROW_STEP   = 30   # every Nth headset sample
ARROW_LENGTH = 0.4  # world units

with_head = [(n, d) for n, d in data.items() if not d['headset'].empty]
if not with_head:
    print('No headset data.')
else:
    for name, d in with_head:
        fig, axes = make_room_fig()
        study = d['study']
        head  = d['headset'].iloc[::ARROW_STEP].copy()
        head['fx'], head['fz'] = zip(*head.apply(
            lambda r: quat_forward_xz(r.rot_x, r.rot_y, r.rot_z, r.rot_w), axis=1))

        for ax, rid in zip(axes, SORTED_ROOMS):
            draw_bg_room(ax, rid)
            rb = ROOM_BOUNDS[rid]
            mask = (head['pos_z'] >= rb['zmin']) & (head['pos_z'] <= rb['zmax'])
            sub  = head[mask]
            if not sub.empty:
                # Axes are (Z horizontal, X vertical) — swap pos and direction components
                ax.quiver(
                    sub['pos_z'].values, sub['pos_x'].values,
                    sub['fz'].values,    sub['fx'].values,
                    color='#00BCD4', alpha=0.6, scale=1/ARROW_LENGTH,
                    scale_units='xy', angles='xy', width=0.006, zorder=3
                )
            ax.set_title(rid, fontsize=8)
            ax.set_xlabel('Z', fontsize=7)
            if ax is axes[0]:
                ax.set_ylabel('X', fontsize=7)
            else:
                ax.set_yticklabels([]); ax.tick_params(left=False)

        pid  = study['participant_id'].dropna().iloc[0] if not study.empty else '?'
        cond = study['condition'].dropna().iloc[0]      if not study.empty else '?'
        fig.suptitle(f'{name}  P={pid}  cond={cond} — Gaze Direction (every {ARROW_STEP} frames)',
                     fontsize=10, y=1.02)
        plt.savefig(DATA_DIR / 'minimap_headset_gaze.png', dpi=150, bbox_inches='tight')
        plt.show()

## Event Timeline

In [ ]:
import subprocess, sys as _sys

timeline_out = DATA_DIR / 'timeline.png'
result = subprocess.run(
    ['uv', 'run', str(SCRIPTS_DIR / 'timeline.py'), str(DATA_DIR), '--out', str(timeline_out)],
    capture_output=True, text=True, cwd=str(SCRIPTS_DIR)
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=_sys.stderr)
if timeline_out.exists():
    display(IPyImage(str(timeline_out)))

## Export CSVs

In [ ]:
for df, name in [
    (overview,   'summary.csv'),
    (latency_df, 'latency.csv') if not latency_df.empty else (None, None),
    (dur_df,     'room_durations.csv') if not dur_df.empty else (None, None),
]:
    if df is None:
        continue
    out = DATA_DIR / name
    df.to_csv(out, index=False)
    print(f'Saved → {out}')